In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)


file_path = "PASTE_YOUR_DATASET_PATH_HERE"

df = pd.read_csv(
    file_path,
    na_values=["?", " ?"],
    skipinitialspace=True
)

print("First 5 rows:")
print(df.head())

print("\nDataset Shape:", df.shape)


df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(".", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

df = df.rename(columns={
    "class": "income",
    "salary": "income"
})

if "income" not in df.columns:
    raise ValueError("Income target column not found!")

df["income"] = (
    df["income"].astype(str)
    .str.strip()
    .str.replace(".", "", regex=False)
)

df["income"] = df["income"].map({
    ">50K": 1,
    "<=50K": 0,
    "1": 1,
    "0": 0
})

df = df.dropna(subset=["income"])


X = df.drop("income", axis=1)
y = df["income"].astype(int)

print("\nClass Distribution:")
print(y.value_counts())

numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()


numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))


model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(
        criterion="gini",
        max_depth=5,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("\nDecision Tree Model Trained Successfully!")


y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["<=50K", ">50K"]
))


y_prob = model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

auc_score = roc_auc_score(y_test, y_prob)

print("\nAUC Score:", auc_score)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr, tpr,
    label=f"Decision Tree (AUC = {auc_score:.3f})"
)

plt.plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="red",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Decision Tree")
plt.legend()
plt.grid()
plt.show()


tree_model = model.named_steps["classifier"]

feature_names = model.named_steps[
    "preprocessing"
].get_feature_names_out()

plt.figure(figsize=(20, 10))

plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=["<=50K", ">50K"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)

plt.title("Decision Tree Visualization")
plt.show()
